# Parameter Estimation
# Exercise 1

## Introduction:
We have a dataset containing the weights of 300 individuals of an insect species, measured at five fixed ages:
$$t=[0, 2.5, 5, 7.5, 10]   days$$

In [1]:
import numpy as np
import pandas as pd
from cffi import VerificationError
from scipy.optimize import curve_fit

# 1. Load the dataset (3000 rows x 5 columns)
# Each row represents an insect, each column represents a measurement time.
data=np.loadtxt('Ex1_dataset (1).txt')

# Fixed measurement time point (days)
t_values=np.array([0, 2.5, 5, 7.5, 10])

# Flatten weights into a single 1D array of 15,000 observations
N_insects, K_times = data.shape
y = data.flatten()

# Repeat time points to align with the flattened observation vector
t_all = np.repeat(t_values,N_insects)

## Model:
The goal is to model the mean weight $w(t)$ as a function of $t$ using:
$$w(t)=a_0+a_1 t^3 + a_2\ln(t+1)$$

Although $w(t)$ is non-linear with respect to $t$, it is linear with respect to the parameters $(a_0, a_1, a_2)$. So, the parameter vector $\hat{a}$ can be calculated analytically using the LS Matrix.

In [4]:
# 2. Analytical least square solution
# Construct Design Matrix X (15,000 x 3)
# Col 0: Intercept term(a0)
# Col 1: t^3 term (a1)
# Col 2: ln(t+1) term (a2)
X = np.column_stack([
    np.ones_like(t_all),
    t_all**3,
    np.log(t_all+1)
])

## LS Estimator
We use the LS method to determine the parameter values that minimize the sum of the squared residuals. We define the residual as:
$$r_i=w_i-[a_o+a_1t_{i}^3+a_2ln(t_i+1)]$$
The quantity that we minimize is the sum of squared residuals:
$$\chi^2 = \sum{[w_i-a_0-a_1t_1^3-a_2ln(t+1)]^2}$$
The best-fit parameters are the values of $a_0$, $a_1$ and $a_2$ that minimize this quantity.
In matrix notation, the model is:
$$w=Xa+\epsilon$$
And the analytical least-square estimator is:
$$\hat{a}=(X^TX)^{-1}X^Tw$$

In [7]:
# Solve for parameter estimates: a_hat = (X^T * X)^(-1) * X^T *y
a_hat = np.linalg.inv(X.T @ X) @ X.T @ y

## Uncertainties of the parameters
The residual variance can be estimated from:

$$s^2=\frac{1}{N-p}\sum_{i=1}^{N}r_i^2,$$

where (N) is the number of individual measurements and (p=3) is the number of fitted parameters. The covariance matrix of the parameter estimates is then:

$$\boxed{C=s^2(X^TX)^{-1}}$$

The std of each parameter estimate is obtained from the square root of the corresponding diagonal element of the covariance matrix:
$$\sigma_{a_0}=\sqrt{C_{00}},\quad $$
$$\sigma_{a_1}=\sqrt{C_{11}},\quad$$
$$\sigma_{a_2}=\sqrt{C_{22}},\quad$$

In [6]:
# Residuals and unbiased noise variance estimate (sigma^2)
residuals = y - X @ a_hat
M = len(y) # Total measurement (15,000)
p = X.shape[1] # Number of parameters (3)
d_o_f = M - p
sigma_sq = np.sum(residuals**2)/d_o_f

# Parameter covariance matrix and standard errors
cov_matrix = sigma_sq * np.linalg.inv(X.T @ X)
param_error = np.sqrt(np.diag(cov_matrix))
print("ANALYTICAL RESULTS:")
print(f"a0: {a_hat[0]: .6f} ± {param_error[0]: .6f}")
print(f"a1: {a_hat[1]: .6f} ± {param_error[1]: .6f}")
print(f"a2: {a_hat[2]: .6f} ± {param_error[2]: .6f}\n")

ANALYTICAL RESULTS:
a0:  12.942049 ±  0.072526
a1: -0.009884 ±  0.000132
a2:  5.024259 ±  0.058209



## Results
$$\boxed{a_0=12.942049 ± 0.072526}$$
$$\boxed{a_1=-0.009884 ± 0.000132}$$
$$\boxed{a_2=5.024259 ± 0.058209}$$


## Covariance matrix
It describes the uncertainties of the fitted parameters and how the parameters are correlated with each other.
$$C=
\begin{pmatrix}
C_{00}&C_{01}&C_{02}\
C_{10}&C_{11}&C_{12}\
C_{20}&C_{21}&C_{22}
\end{pmatrix}$$

The diagonal elements correspond to the variances:
$$C_{00}=\sigma_{a0}^2$$
$$C_{11}=\sigma_{a1}^2$$
$$C_{22}=\sigma_{a2}^2$$


## Verification
As a check of the analytical least-squares solution, the model was also fitted numerically. The numerical results are consistent with the analytical estimates.

In [8]:
# 3. Verification
def model_func(t, a0, a1, a2):
    return a0 + a1* (t**3) + a2 * np.log(t+1)

# Numerical optimization verification
popt, pcov = curve_fit(model_func, t_all, y)
perr = np.sqrt(np.diag(pcov))

print("VERIFICATION:")
print(f"a0: {popt[0]: .6f} (Difference: {abs(a_hat[0] - popt[0]):.2e})")
print(f"a1 : {popt[1]:.6f} (Difference: {abs(a_hat[1] - popt[1]):.2e})")
print(f"a2 : {popt[2]:.6f} (Difference: {abs(a_hat[2] - popt[2]):.2e})\n")


VERIFICATION:
a0:  12.942049 (Difference: 2.63e-11)
a1 : -0.009884 (Difference: 3.69e-13)
a2 : 5.024259 (Difference: 5.87e-11)



## Conclusion
The weight evolution of the insects was modeled using:
$$\boxed{w(t)= a_0+a_1t^3+a_2ln(t+1)}$$
